# 03f — Pre-Flight Risk Model (Problem C)

Train and evaluate the LightGBM binary classifier for pre-flight risk scoring.

**Approach:**
- Temporal split (Train 2018, Val 2019, Test 2020)
- LightGBM binary classification
- Isotonic calibration
- Evaluation via ROC-AUC, PR-AUC, and Lift chart

In [ ]:
import os
from pathlib import Path

root = Path.cwd()
while not (root / "pyproject.toml").exists():
    root = root.parent
os.chdir(root)
print(f"Working directory: {root}")

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

IN_PATH = Path('data/processed/preflight_features_final.parquet')
df = pd.read_parquet(IN_PATH)

features = [
    'month_sin', 'month_cos', 'dow_sin', 'dow_cos', 'hour', 
    'temp', 'rhum', 'prcp', 'wspd', 'pres',
    'airport_risk_rate', 'carrier_risk_rate', 'route_risk_rate',
    'DISTANCE'
]

target = 'incident'

print(f"Dataset: {len(df):,} rows, {len(features)} features.")

## 1. Temporal Split

In [ ]:
df['year'] = pd.to_datetime(df['FL_DATE']).dt.year

train_df = df[df['year'] == 2018]
val_df   = df[df['year'] == 2019]
test_df  = df[df['year'] == 2020]

X_train, y_train = train_df[features], train_df[target]
X_val, y_val     = val_df[features], val_df[target]
X_test, y_test   = test_df[features], test_df[target]

print(f"Train (2018): {len(X_train):,} rows, {y_train.sum()} incidents")
print(f"Val   (2019): {len(X_val):,} rows, {y_val.sum()} incidents")
print(f"Test  (2020): {len(X_test):,} rows, {y_test.sum()} incidents")

## 2. Train LightGBM

In [ ]:
model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    is_unbalance=True,  # Crucial for case-control / imbalanced data
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=50)]
)

print("Model training complete.")

## 3. Calibration

Since we used `is_unbalance=True` and case-control sampling, the raw probabilities are distorted. We must calibrate to get realistic risk scores.

In [ ]:
calibrated_model = CalibratedClassifierCV(model, method='isotonic', cv='prefit')
calibrated_model.fit(X_val, y_val)

plt.figure(figsize=(8, 8))
CalibrationDisplay.from_estimator(calibrated_model, X_test, y_test, n_bins=10, ax=plt.gca())
plt.title("Calibration Plot (Test Set)")
plt.show()

## 4. Evaluation

In [ ]:
y_prob = calibrated_model.predict_proba(X_test)[:, 1]
y_pred = calibrated_model.predict(X_test)

print(f"ROC-AUC:         {roc_auc_score(y_test, y_prob):.4f}")
print(f"Average Precision: {average_precision_score(y_test, y_prob):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

## 5. Feature Importance (SHAP)

In [ ]:
import shap

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test.sample(1000, random_state=42))

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test.sample(1000, random_state=42), plot_type="bar")